# Imports

<a href="https://colab.research.google.com/github/NoaTal1996/Backbone-Graph/blob/main/src/Topic_Clustring_and_NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# try:
#     # Running inside Google Colab → use !pip
#     import google.colab
#     in_google_colab = True
#     print("Running in Colab → installing packages...")
#     !pip install -q bertopic
#     !pip install -q tweet-preprocessor
#     !pip install -q spacy-entity-linker
#     !pip install -q emoji
#     !pip install sentence-transformers scikit-learn
#     !pip install "transformers>=4.40.0,<5" sentencepiece accelerate torch
#     !pip install langdetect
# except ImportError:
#     # Not running in Colab
#     in_google_colab = False
#     pass

In [2]:
import pandas as pd
import pyarrow  # ensures pandas can read/write parquet
import matplotlib.pyplot as plt
import numpy as np

import re
import emoji

from sentence_transformers import SentenceTransformer

from tqdm.auto import tqdm
from IPython.display import display
from langdetect import detect

from pathlib import Path
from collections import defaultdict, deque
from tqdm import tqdm
import json

import matplotlib.ticker as ticker

# Parameters

In [ ]:
is_test = False

In [4]:
input_data_path = "../data/Labeled_Datasets/Cuba/Cuba_part_all.gzip.parquet"
entity_cols_lst = ["hashtags", "account_mentions", "urls", "ner_entities"]

required_columns = [
    "postid",
    "accountid",
    "post_time",
    "post_language",
    "post_text",
    "is_control",
    "hashtags",
    "account_mentions",
    "urls",
]

# Loading and Exporting
[Labeled Datasets for Research on Information Operations](https://zenodo.org/records/14189193)

## Load Data

In [5]:
from pathlib import Path


def strip_data_suffix(path: str | Path) -> str:
    """Return a data filename without its parquet compression suffix.

    Args:
        path: Data file path.

    Returns:
        Filename without a recognized data suffix.
    """
    name = Path(path).name
    for suffix in (".gzip.parquet", ".snappy.parquet", ".parquet", ".gzip", ".gz"):
        if name.endswith(suffix):
            return name[:-len(suffix)]
    return Path(name).stem


input_data_path = Path(input_data_path).expanduser()
data_index = max(
    index for index, part in enumerate(input_data_path.parts) if part == "data"
)
relative_input_path = Path(*input_data_path.parts[data_index + 1:])
dataset_part = strip_data_suffix(relative_input_path)
output_path = (
    Path("../results")
    / relative_input_path.parent
    / dataset_part
    / "step_1_translate_topics_entities"
)
if is_test:
    output_path = output_path / "test"
output_path.mkdir(parents=True, exist_ok=True)
output_data_path = output_path / f"{dataset_part}_step_1.gzip.parquet"

print(f"Loading data from: {input_data_path}")
print(f"Output path: {output_path}")
df = pd.read_parquet(input_data_path, columns=required_columns)
print(f"Loaded required input columns: {required_columns}")

Loading data from: ../data/Labeled_Datasets/Cuba/Cuba_part_all.gzip.parquet
Output path: ../results/Labeled_Datasets/Cuba/Cuba_part_all/step_1_translate_topics_entities/test
Loaded required input columns: ['postid', 'accountid', 'post_time', 'post_language', 'post_text', 'is_control', 'hashtags', 'account_mentions', 'urls']


In [6]:
df["is_control"] = (
    df["is_control"]
    .map({
        True: True,
        False: False,
        1: True,
        0: False,
        "1": True,
        "0": False,
        "True": True,
        "False": False,
        "true": True,
        "false": False,
    })
    .astype("boolean")
)

## Output Path


In [7]:
print(f"Output data path: {output_data_path}")

Output data path: ../results/Labeled_Datasets/Cuba/Cuba_part_all/step_1_translate_topics_entities/test/Cuba_part_all_step_1.gzip.parquet


In [8]:
df.head()

,postid,accountid,post_time,post_language,post_text,is_control,hashtags,account_mentions,urls
0,b9119d7054bbf71d6663dec61a9a6f1bac4c92cc43,bef05ca97a5bcbedc9b6d5e93acb0f7a9389f503b6,2020-01-23 20:32:00,es,#Cuba desenmascara campaña de la contrarrevolu...,False,[Cuba],[c9ff11db289879e3317eebde1aee0cae1fc1dcadcd],[]
1,49817f42052eca195806d4fec976d07f77e6e6d537,bef05ca97a5bcbedc9b6d5e93acb0f7a9389f503b6,2020-01-23 20:33:00,es,@13d032f4fecc73d7abd5fa69fc26f3e9cc06eb9955 Sa...,False,[],"[c9ff11db289879e3317eebde1aee0cae1fc1dcadcd, 1...",[]
2,8392312c7147137cfc0e946a95cd1850ee26b4cc4a,bef05ca97a5bcbedc9b6d5e93acb0f7a9389f503b6,2020-01-23 20:33:00,es,@66ced5291f527da47b64b4864166c8e0cf67b1307f @2...,False,[],"[c9ff11db289879e3317eebde1aee0cae1fc1dcadcd, 6...",[]
3,d16b4aca3b00a3cd0e46b71b39bd45937f2f81f71c,bef05ca97a5bcbedc9b6d5e93acb0f7a9389f503b6,2020-01-23 20:33:00,es,@66ced5291f527da47b64b4864166c8e0cf67b1307f @7...,False,[],"[c9ff11db289879e3317eebde1aee0cae1fc1dcadcd, 6...",[]
4,a78e51ce05e5e92b090eb98fccbd953176b3f574bd,bef05ca97a5bcbedc9b6d5e93acb0f7a9389f503b6,2020-01-23 20:33:00,es,La contrarrevolución cegada x su odio ha comet...,False,[],[c9ff11db289879e3317eebde1aee0cae1fc1dcadcd],[]


## Logs

In [9]:
from functools import wraps
from datetime import datetime



REPORT_PATH = output_path / "progress_report.txt"

def write_report(msg: str):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"{timestamp} | {msg}"

    print(line)
    with open(REPORT_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")


write_report(f"Input data path: {input_data_path}")
write_report(f"Output path: {output_path}")

def report_done(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            result = func(*args, **kwargs)
            write_report(f"Completed: {func.__name__}")
            return result
        except Exception as e:
            write_report(f"FAILED: {func.__name__} | {e}")
            write_report("#######################################################")
            write_report("#######################################################")
            raise
    return wrapper

2026-07-23 15:13:26 | Input data path: ../data/Labeled_Datasets/Cuba/Cuba_part_all.gzip.parquet
2026-07-23 15:13:26 | Output path: ../results/Labeled_Datasets/Cuba/Cuba_part_all/step_1_translate_topics_entities/test


# GPU usage

In [10]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
pipeline_device = 0 if torch.cuda.is_available() else -1
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

if torch.cuda.is_available():
    # Small GPU operation TO validate GPU and signal cluster that the code use GPU.
    warmup_tensor = torch.randn(
        (2048, 2048),
        device=device,
        dtype=torch_dtype,
    )
    warmup_result = warmup_tensor @ warmup_tensor
    torch.cuda.synchronize()

    del warmup_tensor, warmup_result
    torch.cuda.empty_cache()

write_report(f"Using device: {device}")

2026-07-23 15:13:26 | Using device: cpu


# Is Test

In [11]:
if is_test:
    df = df.head(100)
    write_report("Running in test mode: only processing first 100 rows of the dataset.")
    write_report(f"Storing output data to: {output_data_path}")

2026-07-23 15:13:26 | Running in test mode: only processing first 1000 rows of the dataset.
2026-07-23 15:13:26 | Storing output data to: ../results/Labeled_Datasets/Cuba/Cuba_part_all/step_1_translate_topics_entities/test/Cuba_part_all_step_1.gzip.parquet


## Save df

In [12]:
def save_df(df):
    df.to_parquet(output_data_path, index=False, compression="gzip")
    write_report(f"df is saved to: {output_data_path}")


# Dataset Statistics

In [13]:
total_posts = len(df)
hashtag_posts = df[df['hashtags'].notna() & (df['hashtags'].str.len() > 0)]
mention_posts = df[df['account_mentions'].notna() & (df['account_mentions'].str.len() > 0)]
url_posts = df[df['urls'].notna() & (df['urls'].str.len() > 0)]
start_date = df["post_time"].min().strftime("%Y-%m")
end_date = df["post_time"].max().strftime("%Y-%m")

stats_text = [
    f"Posts time range: {start_date} → {end_date}",
    "",
    f"{'df shape:':<40} {df.shape}",
    f"{'df posts with hashtags # shape:':<40} {hashtag_posts.shape}  ({len(hashtag_posts)/total_posts*100:.2f}%)",
    f"{'df posts with account_mentions @ shape:':<40} {mention_posts.shape}  ({len(mention_posts)/total_posts*100:.2f}%)",
    f"{'df posts with urls shape:':<40} {url_posts.shape}  ({len(url_posts)/total_posts*100:.2f}%)",
]

# print to console
print("\n".join(stats_text))

# save to file
if output_path:
    output_path = Path(output_path)
    report_path = output_path / "dataset_summary.txt"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("\n".join(stats_text))

print()
print()

print(f"{'**collum name**':<30} **value type**")
for col in df.columns:
    # dtype_str = str(df[col].dtype)  # Convert dtype to string
    # get a sample non-null value if exists
    sample_val = df[col].dropna().iloc[0] if df[col].notna().any() else None
    print(f"{col:<30} | {type(sample_val).__name__}")


print()
print()
print(df['post_language'].value_counts())

Posts time range: 2020-01 → 2020-01

df shape:                                (1000, 9)
df posts with hashtags # shape:          (541, 9)  (54.10%)
df posts with account_mentions @ shape:  (971, 9)  (97.10%)
df posts with urls shape:                (111, 9)  (11.10%)


**collum name**                **value type**
postid                         | str
accountid                      | str
post_time                      | Timestamp
post_language                  | str
post_text                      | str
is_control                     | bool
hashtags                       | ndarray
account_mentions               | ndarray
urls                           | ndarray


post_language
es     958
und     22
en      15
ca       4
bn       1
Name: count, dtype: int64


### Postid Duplication

In [14]:
postid_duplication_sum = df["postid"].duplicated().sum()
write_report(
    f"postid duplication sum {postid_duplication_sum} "
    f"({postid_duplication_sum / len(df) * 100:.3f}%)"
)

2026-07-23 15:13:27 | postid duplication sum 0 (0.000%)


# Clean Text

## Clean text With Entities

In [15]:
def clean_url(url_match):

    url = url_match.group(0)

    # 1. Remove protocol (http:// or https://)
    url = re.sub(r"https?://", "", url)
    url = re.sub(r"www\.", "", url)

    # 2. Remove query parameters (?x=1)
    url = url.split("?")[0]

    # 3. Split into components by "/"
    parts = url.split("/")

    # Domain (first part)
    domain = parts[0].replace(".", "_")  # twitter.com → twitter_com

    # First section (second part of the URL)
    if len(parts) > 1 and parts[1].strip() != "":
        section = parts[1].replace(".", "_")
        token = f" URL_{domain}_{section} "
    else:
        token = f" URL_{domain} "

    return token

In [16]:
def clean_text(text: str) -> str:

    # lowercase
    text = str(text).lower()

    # URLs → domian + first section
    text = re.sub(r"(https?://\S+|www\.\S+)", clean_url, text)

    # Hashtags → HASHTAG_xxx
    text = re.sub(r"#(\w+)", r" HASHTAG_\1 ", text)

    # Mentions → MENTION_xxx
    text = re.sub(r"@(\w+)", r" MENTION_\1 ", text)

    # emoji → EMOJI_xxx
    text = emoji.demojize(text, delimiters=(" EMOJI_", " "))

    # Collapse spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [17]:
df["clean_text_with_entities"] = df["post_text"].apply(clean_text)

## Clean Text Without Enttities

In [18]:
import re
import pandas as pd

URL_RE = re.compile(r"(https?://\S+|www\.\S+)", flags=re.IGNORECASE)
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")
WHITESPACE_RE = re.compile(r"\s+")

def clean_enteties(text: str) -> str:
    """
    Lightly clean a social-media post while keeping it natural
    for machine translation.

    Operations:
    - Replace URLs with <URL>
    - Replace @username with <USER>
    - Turn #hashtag into 'hashtag' (drop the #)
    - Collapse repeated whitespace
    """
    if not isinstance(text, str):
        text = str(text)

    # lowercase
    text = str(text).lower()

    # 1. Normalize URLs
    text = URL_RE.sub("<URL>", text)

    # 2. Normalize mentions
    text = MENTION_RE.sub("<USER>", text)

    # 3. Normalize hashtags: "#Ecuador2025" -> "Ecuador2025"
    text = HASHTAG_RE.sub(r"\1", text)

    # 4. Collapse whitespace
    text = WHITESPACE_RE.sub(" ", text).strip()

    return text

In [19]:
df["clean_text_without_enteties"] = df["post_text"].astype(str).apply(clean_enteties)

In [20]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 2000)

df[["post_text", "clean_text_without_enteties"]].sample(n=20)

,post_text,clean_text_without_enteties
516,"Una de las primeras acciones de la #RevoluciónCubana fue la creación, el #23deenero de 1960, del Servicio Médico Rural…","una de las primeras acciones de la revolucióncubana fue la creación, el 23deenero de 1960, del servicio médico rural…"
583,#VandalosDelImperio sin ningún escrúpulo asumieron lo que hicieron ... Asumieron ser delicuentes drogadictos .... De l…,vandalosdelimperio sin ningún escrúpulo asumieron lo que hicieron ... asumieron ser delicuentes drogadictos .... de l…
742,"Llueve aún, calle mojada\nVerde renacen las hojas\nY el viento otra vez deshoja\nA una flor enamorada \nQ se quedó hipnotizad…","llueve aún, calle mojada verde renacen las hojas y el viento otra vez deshoja a una flor enamorada q se quedó hipnotizad…"
9,@66ced5291f527da47b64b4864166c8e0cf67b1307f @895ae8b4a7c0de5518a1a097531bab737a10a0e1f8 @76c8692dbf6fececd141a5f730f947a1dbecb3b2c3 @13d032f4fecc73d7abd5fa69fc26f3e9cc06eb9955 @d38a25b043f71f4a2c8dc8e69e7fd0f6a406496feb @48df885f5be7b3e19346f4ca0e0b7f9a7ee3b67080 @021e2e1a09164330203658401d0ea19de31e6427c5 @c8d51d13ae9ecd8541854be1b1fa0ae7590424f6cc @f0d98ebd7eedadd8551ca439a610b22b1ffaf1a7e3 @…,<USER> <USER> <USER> <USER> <USER> <USER> <USER> <USER> <USER> @…
399,@66ced5291f527da47b64b4864166c8e0cf67b1307f @25eb196f32e2323769a552b312b6b000f3b801cdaa @48df885f5be7b3e19346f4ca0e0b7f9a7ee3b67080 @c8d51d13ae9ecd8541854be1b1fa0ae7590424f6cc @13d032f4fecc73d7abd5fa69fc26f3e9cc06eb9955 @c966d67c6211bbac6f3338aaae7bdf261a2c410fd5 @76c8692dbf6fececd141a5f730f947a1dbecb3b2c3 @895ae8b4a7c0de5518a1a097531bab737a10a0e1f8 @e0d024e6e07553a754d71979865523d595dbed56e8 @Frank…,<USER> <USER> <USER> <USER> <USER> <USER> <USER> <USER> <USER> <USER>…
592,"Se estima q hasta 5 mil personas se unieron a la caravana de migrantes q partió de #Honduras el pasado 15 de enero, la prime…","se estima q hasta 5 mil personas se unieron a la caravana de migrantes q partió de honduras el pasado 15 de enero, la prime…"
365,"El bloqueo no solo transgrede los derechos humanos del pueblo cubano, sino el derecho soberano de todos los países del mundo a relacionarse con #Cuba sin temor a ser castigados. | #ElBloqueoEsReal #NoMásBloqueo https://5368f1c3e36bc26ec0c0f332ac2eab6b393a315e56","el bloqueo no solo transgrede los derechos humanos del pueblo cubano, sino el derecho soberano de todos los países del mundo a relacionarse con cuba sin temor a ser castigados. | elbloqueoesreal nomásbloqueo <URL>"
12,José Daniel Ferrer antes de su actividad al servicio de #EEUU tenía una trayectoria delincuencial y de conducta violen…,josé daniel ferrer antes de su actividad al servicio de eeuu tenía una trayectoria delincuencial y de conducta violen…
521,"La Estancia produce jugos y néctares derivados del tomate y frutas, así como Lácteos y otras bebidas, envasados acépti…","la estancia produce jugos y néctares derivados del tomate y frutas, así como lácteos y otras bebidas, envasados acépti…"
693,"Vende Patria sí, antes debieron haberles vendido sus almas al diablo, más que eso son unos desalmados #VándalosDelImperi…","vende patria sí, antes debieron haberles vendido sus almas al diablo, más que eso son unos desalmados vándalosdelimperi…"


# Translation to English

## Language Codes 

In [21]:
# map post_language codes to NLLB-200 language codes
# https://dl-translate.readthedocs.io/en/latest/available_languages/

def map_to_nllb_code(lang_code: str, text: str, undefined_code = "und") -> str:
    """
    Map your dataset's post_language codes to NLLB-200 language codes.
    If not in map, auto-detect the language from text.
    Args:
        lang_code (str): Post language code.
        text (str): Post text for detection.
        undefined_code (str): Default code to return if no mapping is found.
    Returns:
        str: NLLB-200 language code.
    """
    nllb_mapping = {

    # 🌍 Africa
    "af": "afr_Latn",   # Afrikaans
    "am": "amh_Ethi",   # Amharic
    "sw": "swh_Latn",   # Swahili
    "so": "som_Latn",   # Somali

    # 🌍 Middle East
    "ar": "arb_Arab",   # Arabic (MSA)
    "he": "heb_Hebr",   # Hebrew
    "fa": "pes_Arab",   # Persian (Farsi)
    "ur": "urd_Arab",   # Urdu
    "tr": "tur_Latn",   # Turkish

    # 🌍 Europe
    "be": "bel_Cyrl",   # Belarusian
    "bg": "bul_Cyrl",   # Bulgarian
    "bs": "bos_Latn",   # Bosnian
    "ca": "cat_Latn",   # Catalan
    "cs": "ces_Latn",   # Czech
    "cy": "cym_Latn",   # Welsh
    "da": "dan_Latn",   # Danish
    "de": "deu_Latn",   # German
    "el": "ell_Grek",   # Greek
    "en": "eng_Latn",   # English
    "es": "spa_Latn",   # Spanish
    "et": "est_Latn",   # Estonian
    "fi": "fin_Latn",   # Finnish
    "fr": "fra_Latn",   # French
    "ga": "gle_Latn",   # Irish
    "gl": "glg_Latn",   # Galician
    "hr": "hrv_Latn",   # Croatian
    "hu": "hun_Latn",   # Hungarian
    "is": "isl_Latn",   # Icelandic
    "it": "ita_Latn",   # Italian
    "lt": "lit_Latn",   # Lithuanian
    "lv": "lvs_Latn",   # Latvian
    "mk": "mkd_Cyrl",   # Macedonian
    "mt": "mlt_Latn",   # Maltese
    "nl": "nld_Latn",   # Dutch
    "no": "nob_Latn",   # Norwegian (Bokmål)
    "pl": "pol_Latn",   # Polish
    "pt": "por_Latn",   # Portuguese
    "ro": "ron_Latn",   # Romanian
    "ru": "rus_Cyrl",   # Russian
    "sk": "slk_Latn",   # Slovak
    "sl": "slv_Latn",   # Slovenian
    "sq": "als_Latn",   # Albanian
    "sr": "srp_Cyrl",   # Serbian (Cyrillic default)
    "sv": "swe_Latn",   # Swedish
    "uk": "ukr_Cyrl",   # Ukrainian

    # 🌍 Caucasus & Central Asia
    "az": "azj_Latn",   # Azerbaijani
    "hy": "hye_Armn",   # Armenian
    "ka": "kat_Geor",   # Georgian
    "kk": "kaz_Cyrl",   # Kazakh
    "mn": "mon_Cyrl",   # Mongolian
    "uz": "uzn_Latn",   # Uzbek

    # 🌍 South Asia
    "bn": "ben_Beng",   # Bengali
    "gu": "guj_Gujr",   # Gujarati
    "hi": "hin_Deva",   # Hindi
    "kn": "kan_Knda",   # Kannada
    "ml": "mal_Mlym",   # Malayalam
    "mr": "mar_Deva",   # Marathi
    "ne": "npi_Deva",   # Nepali
    "pa": "pan_Guru",   # Punjabi
    "ta": "tam_Taml",   # Tamil
    "te": "tel_Telu",   # Telugu

    # 🌍 Southeast Asia
    "id": "ind_Latn",   # Indonesian
    "ms": "zsm_Latn",   # Malay
    "th": "tha_Thai",   # Thai
    "tl": "tgl_Latn",   # Tagalog
    "vi": "vie_Latn",   # Vietnamese
    "km": "khm_Khmr",   # Khmer

    # 🌍 East Asia
    "ja": "jpn_Jpan",   # Japanese
    "ko": "kor_Hang",   # Korean
    "zh": "zho_Hans",   # Chinese (Simplified)
}

    # 1. direct mapping
    if lang_code in nllb_mapping:
        return nllb_mapping[lang_code]

    # 2. language detection
    try:
        detected = detect(text)
    except Exception as e:
        write_report(f"Language detection failed: {e} | lang_code={lang_code} | text={text[:100]}")
        return undefined_code  # undetermined

    # 3. mapping detected language

    if detected in nllb_mapping:
        return nllb_mapping[detected]
    else:
        write_report(f"No NLLB mapping for detected language: {detected}")
        return undefined_code  # undetermined

In [22]:
# Change post_language to NLLB-200 codes
undefined_code = "und"

# Before mapping
print("Before mapping to NLLB-200 codes:")
language_value_counts = df["post_language"].value_counts()
print(language_value_counts)
write_report(f"Undefined post language percentage **before** mapping: {language_value_counts.get('und', 0) / len(df) * 100:.2f}%")

# Apply mapping
df["post_language"] = df.apply(lambda row: map_to_nllb_code(row['post_language'], row['clean_text_without_enteties'], undefined_code), axis=1)

# After mapping to NLLB-200 codes:
print("After mapping to NLLB-200 codes:")
language_value_counts = df["post_language"].value_counts()
print(language_value_counts)
write_report(f"Undefined post language percentage **after** mapping: {language_value_counts.get('und', 0) / len(df) * 100:.2f}%")

Before mapping to NLLB-200 codes:
post_language
es     958
und     22
en      15
ca       4
bn       1
Name: count, dtype: int64
2026-07-23 15:13:27 | Undefined post language percentage **before** mapping: 2.20%
After mapping to NLLB-200 codes:
post_language
spa_Latn    965
eng_Latn     17
deu_Latn      8
cat_Latn      7
ita_Latn      2
ben_Beng      1
Name: count, dtype: int64
2026-07-23 15:13:28 | Undefined post language percentage **after** mapping: 0.00%


In [23]:
# Remove post with undefined language
undefined_count = df[df['post_language'] == undefined_code].shape[0]
write_report(f"Removing {undefined_count} posts with undefined language code {undefined_code}")
df = df[df['post_language'] != undefined_code].copy()

2026-07-23 15:13:28 | Removing 0 posts with undefined language code und


## Translation

In [24]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

In [25]:
# load models

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

MODEL_NAME = "facebook/nllb-200-distilled-600M"


pipeline_device = 0 if torch.cuda.is_available() else -1

write_report(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch_dtype,
).to(device)
write_report(f"Loaded {MODEL_NAME} on {device} with dtype {torch_dtype}")

translation_pipeline = pipeline(
    "translation",
    model=model,
    tokenizer=tokenizer,
    device=pipeline_device,
    max_length=512,
    batch_size=16,
)
write_report(f"Translation pipeline created.")

2026-07-23 15:13:29 | Using device: cpu


Device set to use cpu


2026-07-23 15:13:59 | Loaded facebook/nllb-200-distilled-600M on cpu with dtype torch.float32
2026-07-23 15:13:59 | Translation pipeline created.


In [26]:
import torch 

def _chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n], i

@report_done
def transformers_translator(
    df: pd.DataFrame,
    translation_pipe,
    text_col: str = "clean_text_without_enteties",
    lang_col: str = "post_language",
    out_col: str = "translated_post",
    target_lang: str = "eng_Latn",
    pivot_lang: str = "spa_Latn",
    backtranslate_same_lang: bool = True,
    call_batch: int = 64,           # how many texts you feed per pipeline call (NOT internal pipeline batch_size). Use to be 16
    max_new_tokens: int = 512,      # cap generated length (huge VRAM saver)
) -> pd.DataFrame:

    # Sort by text length to reduce padding and improving GPU efficiency.
    df["text_len"] = df[text_col].str.len()
    df = df.sort_values("text_len")

    result_df = pd.DataFrame(index=df.index)
    result_df[out_col] = None

    for src_lang, group in tqdm(df.groupby(lang_col), desc="Translating by language"):
        texts = group[text_col].astype(str).tolist()
        indexes = group.index.to_list()

        translated_all = [None] * len(texts)

        def run_translate(batch_texts, s_lang, t_lang):
            # inference_mode saves memory vs no_grad in many cases
            with torch.inference_mode():
                outs = translation_pipe(
                    batch_texts,
                    src_lang=s_lang,
                    tgt_lang=t_lang,
                    # truncation=True,                # avoid super-long inputs
                    max_new_tokens=max_new_tokens,  # avoid 1024-token generations
                )
            return [o["translation_text"] for o in outs]

        # Case 1: normal translation
        if src_lang != target_lang:
            write_report(f"{len(texts)} posts: {src_lang} → {target_lang}")

            for batch_texts, start in _chunks(texts, call_batch):
                batch_trans = run_translate(batch_texts, src_lang, target_lang)
                translated_all[start:start+len(batch_trans)] = batch_trans

                # cleanup between calls (helps on long runs)
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        # Case 2: back-translation
        elif backtranslate_same_lang:
            write_report(f"{len(texts)} posts: {target_lang} → {pivot_lang} → {target_lang}")

            for batch_texts, start in _chunks(texts, call_batch):
                pivot_texts = run_translate(batch_texts, target_lang, pivot_lang)
                back_texts = run_translate(pivot_texts, pivot_lang, target_lang)
                translated_all[start:start+len(back_texts)] = back_texts

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        # assign back
        result_df.loc[indexes, out_col] = translated_all

    return result_df

In [ ]:
print(f"df shape: {df.shape}")
translated_df = transformers_translator(df, translation_pipeline)
df = df.drop(columns=["translated_post"], errors="ignore") # Drop the old column if exist
df = df.join(translated_df)
write_report("Translation completed.")

df shape: (1000, 11)


Translating by language:   0%|          | 0/6 [00:00<?, ?it/s]

2026-07-23 15:13:59 | 1 posts: ben_Beng → eng_Latn


Translating by language:  17%|█▋        | 1/6 [00:19<01:39, 19.92s/it]

2026-07-23 15:14:19 | 7 posts: cat_Latn → eng_Latn


...

2026-07-23 15:14:58 | 17 posts: eng_Latn → spa_Latn → eng_Latn


Translating by language:  67%|██████▋   | 4/6 [03:09<02:08, 64.05s/it]

2026-07-23 15:17:09 | 2 posts: ita_Latn → eng_Latn


Translating by language:  83%|████████▎ | 5/6 [03:14<00:42, 42.61s/it]

2026-07-23 15:17:14 | 965 posts: spa_Latn → eng_Latn


In [ ]:
pd.set_option("display.width", 3000)
pd.set_option("display.max_colwidth", 1500)

df.sample(10)[["clean_text_without_enteties", "translated_post"]]

## Remove Engllish Stopwords

In [ ]:
from nltk.corpus import stopwords
import nltk

In [ ]:
nltk.download('stopwords')

EXTRA_STOPWORDS = [
    'he', 'she', 'they', 'you', 'i', 'we', 'im', 'hes', 'shes', 'its', 'rt', 'amp', 're',
    'theyre', 'dont', 'doesnt', 'would', 'could', 'like', 'know', 'say', 'said', 'one',
    'look', 'looks', 'think', 'oh', 'yeah', 'guy', 'gonna', 'thing', 'someone', 'people',
    'get', 'got', 'thats', 'just', 'really', 'even', 'still', 'want', 'need', 'needs',
    'me', 'to', 'you', 'the', 'that', 'be', 'my', 'get', 'in', 'and',
     'rt', 'the', 'https', 'to', 'my', 'it', 'on', 'you', 'of', 'for', 'https', 'http', 'RT'
]
all_stopwords_english = set(stopwords.words('english') + EXTRA_STOPWORDS)

def remove_stopwords(text):
    if not text:
        return text
    return " ".join([w for w in text.lower().split() if w not in all_stopwords_english])


In [ ]:
df["clean_text_with_entities"] = df["clean_text_with_entities"].apply(remove_stopwords)
df["clean_text_without_enteties"] = df["clean_text_without_enteties"].apply(remove_stopwords)
df["translated_post"] = df["translated_post"].apply(remove_stopwords)

In [ ]:
df[["post_text", "clean_text_with_entities"]].head(20)

## Check Translation

In [ ]:
print("Untranslated posts samples:")
display(df[df["translated_post"].isna()][["post_text", "post_language", "clean_text_without_enteties", "translated_post"]])

# Check if more than 10% of posts are untranslated
num_untranslated = df["translated_post"].isna().sum()
total_posts = len(df)
percentage_untranslated = (num_untranslated / total_posts) * 100

write_report(f"Untranslated posts: {num_untranslated}/{total_posts} ({percentage_untranslated:.2f}%)")
if percentage_untranslated > 10:
    write_report(f"Handling None Cells: More than 10% of posts are untranslated: {percentage_untranslated:.2f}%")
    raise ValueError(f"More than 10% of posts are untranslated: {percentage_untranslated:.2f}%")

# Fill untranslated posts with original cleaned text
df["translated_post"] = df["translated_post"].fillna(df["clean_text_without_enteties"])
write_report("Filled untranslated posts with original cleaned text.")

## Save df

In [ ]:
save_df(df)

write_report(
    "Finished Translation\n"
    f"Saved df to {output_path}\n"
    "Continuing to Entities Reuse"
)

# Named Entity Recognition (NER)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

NER_MODEL_NAME = "TweebankNLP/bertweet-tb2_wnut17-ner"
ner_device = "cuda" if torch.cuda.is_available() else "cpu"
ner_pipeline_device = 0 if torch.cuda.is_available() else -1
ner_torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(NER_MODEL_NAME)
ner_model = AutoModelForTokenClassification.from_pretrained(
    NER_MODEL_NAME,
    torch_dtype=ner_torch_dtype,
).to(ner_device)
write_report(f"Loaded {NER_MODEL_NAME} on {ner_device} with dtype {ner_torch_dtype}")

ner_pipeline = pipeline(
    "token-classification",
    model=ner_model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=ner_pipeline_device,
    batch_size=32,
)

In [ ]:
def _resolve_ner_entity(raw_word: str, text: str) -> str:
    """Clean NER entity and recover best matching word from text.

    Args:
        raw_word: Raw entity from model.
        text: Original text.

    Returns:
        Cleaned entity string.
    """
    cleaned = raw_word.replace("@@", "").replace("<unk>", "").strip()
    if not cleaned:
        return ""

    # If no artifacts → trust model
    if cleaned == raw_word:
        return cleaned

    text_lower = text.lower()
    cleaned_lower = cleaned.lower()

    # find all occurrences (not just first)
    matches = list(re.finditer(re.escape(cleaned_lower), text_lower))

    if matches:
        best_match = None
        best_len = -1

        for m in matches:
            start, end = m.start(), m.end()

            # expand LEFT
            while start > 0 and text[start - 1].isalnum():
                start -= 1

            # expand RIGHT
            while end < len(text) and text[end].isalnum():
                end += 1

            candidate = text[start:end].strip()

            if len(candidate) > best_len:
                best_match = candidate
                best_len = len(candidate)

        return best_match

    return cleaned

In [ ]:
@report_done
def extract_ner_entities(posts_lst: list[str]) -> list[list[str]]:
    """Run NER on posts and return cleaned entity strings per post.

    Args:
        posts_lst: List of input post texts.

    Returns:
        List of entity lists per post.
    """
    ner_outputs = ner_pipeline(posts_lst)

    all_posts_entities = []
    for post_text, post_ner_output in zip(posts_lst, ner_outputs):
        post_entities = []

        for entity_dict in post_ner_output:
            raw_word = entity_dict["word"]
            cleaned_entity = _resolve_ner_entity(raw_word, post_text)
            if cleaned_entity:
                post_entities.append(cleaned_entity)

        # remove duplicates
        post_entities = list(dict.fromkeys(post_entities))
        # append to results
        all_posts_entities.append(post_entities)

    return all_posts_entities

In [ ]:
# Apply ONCE, no pandas apply, in order to utilize batches.
df["ner_entities"] = extract_ner_entities(df["translated_post"].to_list())

In [ ]:
# display post with ner entities
df_ner_filtered = df.loc[df["ner_entities"].str.len() > 0, ["translated_post", "ner_entities"]]
display(df_ner_filtered.sample(20))

# Clean Entities

In [ ]:
from collections.abc import Iterable

def cast_to_list(value) -> list:
    """Convert a value to a list.

    Args:
        value: Cell value to convert.

    Returns:
        An empty list for missing values, a converted list for iterables,
        or a single-element list for scalar values.
    """
    if value is None:
        return []

    if not isinstance(value, Iterable) or isinstance(value, (str, bytes)):
        if pd.isna(value):
            return []
        return [value]

    return list(value)


@report_done
def clean_entity_columns(
    df: pd.DataFrame,
    entity_cols_lst: list[str],
) -> None:
    """Normalize entity columns to match TfidfVectorizer behavior.

    Converts missing values to empty lists, iterable values to lists, and
    scalar values to single-element lists. Each entity is then converted to
    a stripped, lowercase string with spaces replaced by underscores.

    Args:
        df: Input DataFrame. Modified in place.
        entity_cols_lst: Columns containing entities.

    Returns:
        None.
    """
    for col in entity_cols_lst:
        df[col] = df[col].map(cast_to_list)

        df[col] = df[col].map(
            lambda entities: [
                str(entity).strip().lower().replace(" ", "_")
                for entity in entities
            ]
        )

In [ ]:
clean_entity_columns(df, entity_cols_lst)

In [ ]:
write_report(f"Saved df with clean_entity_columns, entity_cols_lst = {entity_cols_lst}")
save_df(df)

# Entities Reuse

In [ ]:
# Parameters
# entity_cols_lst as defined in the parameters section at the top of the notebook
time_windows = ["12H", "3D"] # See pandas.TimeDelta documentation. Example: "1W" for 1 week, "4D" for 4 days, "12H" for 12 hours, etc.

In [ ]:
@report_done
def add_entity_reuse(
    df: pd.DataFrame,
    time_window: str = "3D",
    entity_column: str = "hashtags",
    post_id_column: str = "postid",
    account_id_column: str = "accountid",
    timestamp_column: str = "post_time",
) -> None:
    """
    Adds temporal entity-reuse columns to a DataFrame.

    For each post, this function finds earlier posts within a `time_window` that share at least one entity.
    It adds columns with the post IDs and account IDs of these related posts.

    It maintains per-entity sliding windows to efficiently track recent usage,
    collects matching prior posts, and stores their post IDs and account IDs in new columns.
    It use dict{entity: deque[(timestamp, post_index)]} to track recent posts for each entity, expiring old entries as it goes.

    Note 1: An authors can reuse entities from their own earlier posts, and these will be included in the results. You can later set to ignore self-reuse when building the graph.
    Note 2: The function does not count for a post that reuses the same entity type multiple times between posts. It will only be counted once per post.

    Args:
        df (pd.DataFrame): The input DataFrame.
        entity_column (str): The column with entities for each post.
        time_window (str): The sliding window duration (default: "4D").
        post_id_column (str): The column with post identifiers (default: "postid").
        account_id_column (str): The column with account identifiers (default: "accountid").
        timestamp_column (str): The column with post timestamps (default: "post_time").

    Returns:
        None: Modifies the DataFrame in-place, adding:
              - f"{entity_column}_reuse_{time_window}_postids"
              - f"{entity_column}_reuse_{time_window}_accountids"
    """

    write_report(f"\n[add_entity_reuse] Adding entity reuse columns for '{entity_column}' with time window {time_window}...")

    # 1. Define output column names
    output_post_ids_column = f"{entity_column}_reuse_{time_window}_postids"
    output_account_ids_column  = f"{entity_column}_reuse_{time_window}_accountids"
    
    # Drop existing columns if they already exist to avoid conflicts
    existing_cols_to_drop = [col for col in [output_post_ids_column, output_account_ids_column] if col in df.columns]
    if existing_cols_to_drop:
        write_report(f"Dropping existing columns: {existing_cols_to_drop}")
        df.drop(columns=existing_cols_to_drop, inplace=True)

    # 2. Convert columns to NumPy arrays for performance
    timestamps = pd.to_datetime(df[timestamp_column]).to_numpy(dtype="datetime64[ns]")
    post_ids = df[post_id_column].to_numpy()
    account_ids = df[account_id_column].to_numpy()
    entities_per_post = df[entity_column].to_numpy()

    # Convert time window to numpy timedelta
    time_window_delta = np.timedelta64(pd.Timedelta(time_window).value, "ns")

    # Pre-allocate output lists
    num_posts = len(df)
    output_post_ids = [[] for _ in range(num_posts)]
    output_account_ids  = [[] for _ in range(num_posts)]

    # 3. Set up entity history tracker: {entity: deque[(timestamp, post_index)]}
    entities_history: dict[str, deque] = defaultdict(deque)

    # 4. Process posts in chronological order
    sorted_timestamp_indices = np.argsort(timestamps)
    for current_post_index in tqdm(sorted_timestamp_indices, desc="Processing posts"):
        current_timestamp = timestamps[current_post_index]
        current_entities = entities_per_post[current_post_index]

        # no entities in this post
        if current_entities is None or (len(current_entities) == 0):
            continue

        matching_post_indices = set()

        # 5. Find related earlier posts for each entity
        for entity in current_entities:
            
            # no entity
            if not entity:
                continue

            entity_post_deque = entities_history[entity] # deque: [(timestamp, post_index)] of post with this entity
            
            # Expire old posts from the deque
            while entity_post_deque and (current_timestamp - entity_post_deque[0][0]) > time_window_delta:
                entity_post_deque.popleft() # remove posts outside the time window

            # Collect indices of posts still in the window
            matching_post_indices.update(post_index for _, post_index in entity_post_deque)

        # 6. Store results in df
        if matching_post_indices:
            sorted_matching_indices = sorted(list(matching_post_indices), key=lambda index: timestamps[index]) # label order by timestamps, oldest is first

            # Remove duplicates while preserving order
            output_post_ids[current_post_index] = list(dict.fromkeys([post_ids[post_index] for post_index in sorted_matching_indices]))
            output_account_ids[current_post_index] = list(dict.fromkeys([account_ids[post_index] for post_index in sorted_matching_indices]))

        # 7. Add current post to history (after matching)
        for entity in current_entities:
            if entity:
                entities_history[entity].append((current_timestamp, current_post_index))

    # 8. Convert outputs to lists and add to df
    # Had issues since this column contains long lists. Parquet ran into OOM with gzip compression and ArrowNotImplementedError when reading it back.
    # df[output_post_ids_column] = [json.dumps(x) for x in output_post_ids]
    # print(f"[add_entity_reuse] Added column: {output_post_ids_column}")

    df[output_account_ids_column] = [list(x) for x in output_account_ids]
    print(f"[add_entity_reuse] Added column: {output_account_ids_column}")

In [ ]:
# entity_cols_lst as defined in the parameters section at the top of the notebook

# Execute entity reuse processing
for time_window in time_windows:
    for entity_col in entity_cols_lst:
        add_entity_reuse(df, time_window=time_window, entity_column=entity_col)

## Validation

In [ ]:
def sample_posts_reuse_entities(
    df: pd.DataFrame,
    reuse_col: str = "hashtags_reuse_3D_postids",
    random_state: int | None = None,
) -> pd.DataFrame:
    """
    Sample one post with non-empty reuse list and return it together
    with all reused posts.

    Args:
        df: Posts dataframe.
        reuse_col: Column containing list of reused postids.
        random_state: Optional seed for reproducibility.

    Returns:
        DataFrame containing:
            - The sampled post
            - All posts referenced in its reuse column
        With selected columns only.
    """

    cols_to_keep = [
        # "postid",
        #"accountid",
        "post_time",
        #"post_text",
        "translated_post",
        # "post_language",
        "hashtags",
        "account_mentions",
        "urls",
        "ner_entities",
        # reuse_col,
    ]

    # Filter non-empty reuse lists
    valid_rows = df[df[reuse_col].apply(lambda x: isinstance(x, list) and len(x) > 0)]

    if valid_rows.empty:
        raise ValueError(f"No posts found with non-empty {reuse_col}")

    # Sample one row
    sample_row = valid_rows.sample(1, random_state=random_state)

    reused_postids = sample_row[reuse_col].iloc[0]

    # Get reused posts
    reused_rows = df[df["postid"].isin(reused_postids)].sort_values("post_time")

    # Combine
    df_reuse_entities = pd.concat([sample_row, reused_rows])[cols_to_keep]

    return df_reuse_entities

In [ ]:
# Had issues since postids column contains long lists. Parquet ran into OOM with gzip compression and ArrowNotImplementedError when reading it back.
# to use this cell, go to `add_entity_reuse`` function and uncomment the line that adds postids column to the df.

# reuse_col = "hashtags_reuse_3D_postids"
# df_reuse_entities = sample_posts_reuse_entities(df, reuse_col=reuse_col)

# with pd.option_context(
#     "display.max_colwidth", None,
#     "display.max_columns", None,
#     "display.width", None
# ):
#     entity_type = reuse_col.split("_reuse_")[0]
#     print(f"The first post is the sampled post with reused {entity_type}, followed by the posts that the post use their {entity_type} within the {time_window}.")
#     display(df_reuse_entities)

## Save df

In [ ]:
save_df(df)

write_report(
    "Finished Entities Reuse\n"
    f"Added reuse columns for entity types: {entity_cols_lst} and time windows: {time_windows}\n"
    "Continuing to Topic Clustering"
)

# K-Means English Topic Clustering

Note: the topic clustering don't influenced by the entities since it is preformed on the text without the entitites.

## Embeddings

In [ ]:
import hashlib
import json

from sentence_transformers import SentenceTransformer

post_text_col = "translated_post"
embeddings_model_name = "digio/Twitter4SSE"

posts_texts = df[post_text_col].astype(str).tolist()


def fingerprint_texts(texts: list[str]) -> str:
    """Create a stable fingerprint for an ordered collection of texts.

    Args:
        texts: Texts in the same order used to generate embeddings.

    Returns:
        SHA-256 fingerprint of the ordered texts.
    """
    digest = hashlib.sha256()
    for text in texts:
        encoded_text = text.encode("utf-8")
        digest.update(len(encoded_text).to_bytes(8, byteorder="little"))
        digest.update(encoded_text)
    return digest.hexdigest()


model = SentenceTransformer(embeddings_model_name, device=device)

embeddings = model.encode(
    posts_texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True, 
)

embeddings = np.ascontiguousarray(embeddings, dtype=np.float32) 

embeddings_path = Path(output_path) / "post_embeddings.npy"
embeddings_metadata_path = Path(output_path) / "post_embeddings_metadata.json"

embeddings_metadata = {
    "model_name": embeddings_model_name,
    "text_column": post_text_col,
    "row_count": len(posts_texts),
    "embedding_dimension": embeddings.shape[1],
    "texts_fingerprint": fingerprint_texts(posts_texts),
}

np.save(embeddings_path, embeddings)
embeddings_metadata_path.write_text(
    json.dumps(embeddings_metadata, indent=2),
    encoding="utf-8",
)

write_report(f"Generated embeddings, from: {post_text_col}, with model: {embeddings_model_name}, shape: {embeddings.shape}")
write_report(f"Saved embeddings to: {embeddings_path}")
write_report(f"Saved embeddings metadata to: {embeddings_metadata_path}")

## K-Means

In [ ]:
if torch.cuda.is_available():
    from cuml.cluster import KMeans
    from cuml.metrics.cluster import silhouette_score
else:
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score

In [ ]:
@report_done
def auto_kmeans_topics(
    embeddings: np.ndarray,
    k_min: int = 4,
    k_max: int = 64,
    step: int = 4,
) -> tuple[np.ndarray, int]:
    """Find the best KMeans cluster count using silhouette score.

    Args:
        embeddings: Embedded texts with shape ``(n_samples, n_features)``.
        k_min: Minimum number of clusters.
        k_max: Maximum number of clusters.
        step: Difference between tested cluster counts.

    Returns:
        Labels from the best model and its cluster count.
    """
    best_k = None
    best_score = float("-inf")
    best_labels = None

    k_max = min(k_max, len(embeddings) - 1)

    for k in range(k_min, k_max + 1, step):
        write_report(f"Trying k={k}")

        model = KMeans(
            n_clusters=k,
            random_state=42,
            n_init="auto",
        )
        labels = model.fit_predict(embeddings)
        score = float(silhouette_score(embeddings, labels))

        write_report(f"k={k}, silhouette_score={score:.4f}")

        if score > best_score:
            best_score = score
            best_k = k

            # Ensure the returned labels are a NumPy array.
            if hasattr(labels, "get"):
                best_labels = labels.get()
            elif hasattr(labels, "to_numpy"):
                best_labels = labels.to_numpy()
            else:
                best_labels = np.asarray(labels)

    write_report(
        f"Best k={best_k}, silhouette_score={best_score:.4f}"
    )

    return best_labels, best_k

In [ ]:
# import math

# n = len(df)

# k_min = max(3,#  int(math.log(n)))
# print(f"k_min = {k_min}")
# k_max = min(200 ,int(math.sqrt(n)))
# print(f"k_max = {k_max}")
# labels, best_k = auto_kmeans_topics(embeddings, k_min=k_min, k_max=k_max)

labels, best_k = auto_kmeans_topics(embeddings, k_min=4, k_max=128, step=4)

k_means_topic_col_name = f"K_Means_topic_{best_k}"
df[k_means_topic_col_name] = labels
df.head()

In [ ]:
save_df(df)
write_report(f"Saved df with K_Means topics. Best k = {best_k}")

## Plots

### Utils

In [ ]:
@report_done
def make_topic_stats(df, topic_col_name):
    """
    Calculate per-topic post counts.

    Args:
        df (pd.DataFrame): Data with topics and 'is_control'.
        topic_col_name (str): Column name of topic labels.

    Returns:
        topic_stats (pd.DataFrame): Colums: [topic, total_count, io_Count, legitimate_posts, io_percent].
    """
    topic_stats = (
        df.groupby(topic_col_name)
          .agg(
              total_count=(topic_col_name, "count"),
              io_count=("is_control", lambda x: (~x).sum())
          )
          .reset_index()
    )
    topic_stats["legitimate_posts"] = topic_stats["total_count"] - topic_stats["io_count"]
    topic_stats["io_percent"] = topic_stats["io_count"] / topic_stats["total_count"] * 100
    return topic_stats

In [ ]:
from matplotlib.lines import Line2D

def plot_topic_stats(topic_stats, topic_col_name, output_path: str | Path = ""):
    """
    Plot stacked IO vs legitimate post counts per topic.
    Saves the fig to output_path if provided.

    Args:
        topic_stats (pd.DataFrame): Output of make_topic_stats().
        topic_col_name (str): Topic column to plot.
        output_path (str): Path to *folder* save the plot.

    Returns:
        None
    """
    topics = topic_stats[topic_col_name]
    io_count = topic_stats["io_count"]
    legitimate_count = topic_stats["legitimate_posts"]
    io_percent = topic_stats["io_percent"]

    plt.figure(figsize=(12, 6))

    # 🔴 Bottom: IO posts
    bar_io = plt.bar(topics, io_count, label="IO posts", color="red")

    # 🔵 Top: Non-IO posts stacked above IO
    bar_legit = plt.bar(topics, legitimate_count, bottom=io_count, label="Non-IO posts")

    # Add IO percentage text
    ax = plt.gca()
    ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))# Format y-axis with commas
    
    for x, io, legit, pct in zip(topics, io_count, legitimate_count, io_percent):
        total_height = io + legit
        ax.text(
            x,
            total_height + 0.5,              # slightly above the bar
            f"{pct:.1f}%",                 # format like 23.5%
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="regular"
        )

    plt.title(f"Post Count per Topic ({topic_col_name})")
    plt.xlabel("Topic ID")
    plt.ylabel("Number of Posts")
    plt.xticks(topics)
    plt.grid(axis="y")

    percent_legend = Line2D([], [], color="none")
    total_posts_legend = Line2D([], [], color="none")
    plt.legend(
        handles=[bar_io, bar_legit, percent_legend, total_posts_legend],
        labels=[
            "IO posts",
            "Non-IO posts",
            "% IO Posts Ratio",
            f"Total posts: {topic_stats['total_count'].sum():,}",
        ],
        loc="best",
    )


    plt.tight_layout()


    if output_path:
        output_path = Path(output_path)
        plot_file_path = output_path / (f"{topic_col_name}_posts_clustering.png")
        plt.savefig(plot_file_path, dpi=300)
        print(f"plot is saved to {plot_file_path}")

    plt.show()

    plt.close()

In [ ]:
@report_done
def make_topic_account_stats(
    df: pd.DataFrame,
    topic_col: str,
    author_col: str = "accountid",
    is_control_col: str = "is_control",
    threshold: float = 0.5,
) -> pd.DataFrame:
    """Compute per-topic unique account counts (IO vs control) using an author-level threshold.

    Args:
        df: Post-level DataFrame.
        topic_col: Topic column name.
        author_col: Author identifier column.
        is_control_col: Column where 1 = control, 0 = IO.
        threshold: IO-rate threshold to label an author as IO (1 = IO).

    Returns:
        DataFrame with per-topic counts of unique IO/control accounts and IO percentage.
    """
    # Author-level IO rate: 1 - mean(is_control)
    author_io_rate = 1 - df.groupby(author_col)[is_control_col].mean()
    author_io_tag = (author_io_rate >= threshold).astype(int)

    topic_author_df = df[[topic_col, author_col]].drop_duplicates().copy()
    topic_author_df["author_io_tag"] = topic_author_df[author_col].map(author_io_tag)

    stats = (
        topic_author_df
        .groupby(topic_col)["author_io_tag"]
        .value_counts()
        .unstack(fill_value=0)
    )

    # Ensure both columns exist even if absent in data
    stats = stats.reindex(columns=[0, 1], fill_value=0).rename(
        columns={0: "control_accounts", 1: "io_accounts"}
    ).reset_index()

    stats["total_accounts"] = stats["control_accounts"] + stats["io_accounts"]
    stats["io_percent"] = (stats["io_accounts"] / stats["total_accounts"]).mul(100)

    return stats

In [ ]:
def plot_topic_account_stats(
    topic_account_stats: pd.DataFrame,
    topic_col_name: str,
    output_path: Path | str | None = None,
):
    """Plot IO vs control account counts per topic.

    Args:
        topic_account_stats: Output of make_topic_account_stats().
        topic_col_name: Topic column name.
        model_name: Model name for title / filename.
        output_path: Optional folder to save the plot.

    Returns:
        None
    """
    topics = topic_account_stats[topic_col_name]
    io_count = topic_account_stats["io_accounts"]
    control_count = topic_account_stats["control_accounts"]
    io_percent = topic_account_stats["io_percent"]

    plt.figure(figsize=(12, 6))

    # 🔴 Bottom: IO accounts
    bar_io = plt.bar(topics, io_count, label="IO accounts", color="red")

    # 🔵 Top: Control accounts
    bar_control = plt.bar(
        topics,
        control_count,
        bottom=io_count,
        label="Control accounts",
    )

    # Add IO percentage text
    ax = plt.gca()
    ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
    
    for x, io, ctrl, pct in zip(topics, io_count, control_count, io_percent):
        total = io + ctrl
        ax.text(
            x,
            total + 0.3,
            f"{pct:.1f}%",
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="regular",
        )

    plt.title(f"Unique Accounts per Topic ({topic_col_name})")
    plt.xlabel("Topic ID")
    plt.ylabel("Number of Unique Accounts")
    plt.xticks(topics)
    plt.grid(axis="y")

    # Legend with extra info lines
    percent_legend = Line2D([], [], color="none")
    total_accounts_legend = Line2D([], [], color="none")

    plt.legend(
        handles=[bar_io, bar_control, percent_legend, total_accounts_legend],
        labels=[
            "IO accounts",
            "Control accounts",
            "% IO Accounts Ratio",
            f"Total accounts: {topic_account_stats['total_accounts'].sum():,}",
        ],
        loc="best",
    )

    if output_path:
        output_path = Path(output_path)
        plot_file_path = output_path / f"{topic_col_name}_accounts_clustering.png"
        plt.savefig(plot_file_path, dpi=300)
        print(f"Plot saved to {plot_file_path}")

    plt.show()
    plt.close()

### Topic Stats

In [ ]:
# topic_col_name as defined in the last cell 
topic_stats = make_topic_stats(df, k_means_topic_col_name)
plot_topic_stats(topic_stats, k_means_topic_col_name, output_path)

### Account Stats

In [ ]:
# topic_col_name as defined in the last cell 
topic_account_stats = make_topic_account_stats(df, k_means_topic_col_name)
plot_topic_account_stats(topic_account_stats, k_means_topic_col_name, output_path)

## Posting Same Topic

In [ ]:
from collections import defaultdict, deque

@report_done
def add_posting_same_topic(
    df: pd.DataFrame,
    topic_column: str,
    time_window: str = "3D",
    account_id_column: str = "accountid",
    timestamp_column: str = "post_time",
) -> str:
    """Add prior authors posting the same topic within a time window.

    The DataFrame is modified in place. Author IDs are unique and ordered by
    the timestamps of their matching posts, from oldest to newest.

    Returns:
        Name of the added account-ID column.
    """
    required_columns = {topic_column, account_id_column, timestamp_column}
    missing_columns = sorted(required_columns - set(df.columns))
    if missing_columns:
        raise KeyError(f"Missing required columns: {missing_columns}")

    output_column = f"posting_same_topic_{topic_column}_{time_window}_accountids"
    timestamps = pd.to_datetime(df[timestamp_column], errors="coerce").to_numpy(
        dtype="datetime64[ns]"
    )
    topics = df[topic_column].to_numpy()
    account_ids = df[account_id_column].to_numpy()
    window_delta = np.timedelta64(pd.Timedelta(time_window).value, "ns")

    output_account_ids = [[] for _ in range(len(df))]
    topic_history: dict[object, deque] = defaultdict(deque)

    for current_index in np.argsort(timestamps, kind="stable"):
        current_timestamp = timestamps[current_index]
        current_topic = topics[current_index]
        if np.isnat(current_timestamp) or pd.isna(current_topic):
            continue

        topic_posts = topic_history[current_topic]
        while topic_posts and current_timestamp - topic_posts[0][0] > window_delta:
            topic_posts.popleft()

        output_account_ids[current_index] = list(
            dict.fromkeys(account_ids[post_index] for _, post_index in topic_posts)
        )
        topic_posts.append((current_timestamp, current_index))

    df[output_column] = output_account_ids
    write_report(
        f"[add_posting_same_topic] Added {output_column} for topic column "
        f"{topic_column} and time window {time_window}."
    )
    return output_column

In [ ]:
posting_same_topic_columns = [
    add_posting_same_topic(
        df=df,
        topic_column=k_means_topic_col_name,
        time_window=time_window,
    )
    for time_window in time_windows
]

# Save and Finish

In [ ]:
# print dataset statistics
for col in df.columns:
    sample_val = df[col].dropna().iloc[0] if df[col].notna().any() else None
    print(f"{col:<30} | {type(sample_val).__name__}")

In [ ]:
output_df = pd.read_parquet(input_data_path)
for column in df.columns:
    write_report(f"Updating column {column} to output_df")
    output_df[column] = df[column]
save_df(output_df)
write_report(f"Saved final output_df to {output_path}")

In [ ]:
write_report("Finished Running Successfully")
write_report("#######################################################")
write_report("#######################################################")